# Lesson 18 (follow-on): Receipts That Prove a *Human* Authorized the Action

Lesson 18 proves what the **agent** did and what the **gate** decided. This notebook adds the missing half: proof that a **named human** approved the **exact** action — a separate, human-held signature over the full canonical action, verified offline against a pinned key.

Both artifacts here — the human **approval receipt** and the agent **action receipt** — share **one envelope shape** (the same shape as the lesson's action receipts, per the co-signature path in draft-farley-acta-signed-receipts), so a single `verify_chain` covers both kinds. What stays separate are the **authorities**: the approval verifies against the pinned *approver* key set, the action receipt against the pinned *agent* key set. One verifier contract, two authorities.

The rule the whole notebook teaches: **a signed approval is not authority by itself.** Authority exists only if the approval receipt and the action receipt still bind to the same canonical action at execution time — under a policy version, key, and expiry that are still current, and an approval that has not already been consumed. If any of that fails, the chain refuses with a **distinct reason**, so you can tell *authority went stale* apart from *the executed action changed*.

In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
import jcs
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import BadSignatureError, CryptoError

def b64u(b):      return base64.urlsafe_b64encode(b).rstrip(b"=").decode()
def b64u_dec(s):  return base64.urlsafe_b64decode(s + "=" * (-len(s) % 4))
def canon(obj):   return jcs.canonicalize(obj)            # RFC 8785 -> bytes
def sha256_hex(b): return hashlib.sha256(b).hexdigest()

## The exact action

The unit of approval is the **canonical action object** — not a vague label like "approve refund," but the precise, fully-specified action. Signing over the whole object (and deriving a digest from it) is what lets us later prove the human approved *this* and nothing else.

In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_version": "refunds-v3",
}
print("action digest:", sha256_hex(canon(action)))

action digest: e68137d4f7788a7df1e64c5fe25a8e61ddb02c05a8a4f80d0cf0a5c642199138


## One envelope, two authorities

Every receipt is `{payload, sig}` where `payload.key_id` names the signing key and the signature covers the canonical payload bytes. `verify_envelope` is the **shared** structural + signature check for both receipt kinds; which *pinned key registry* it resolves `key_id` against is what keeps the authorities separate:

- **approval receipt** — named approver, approval `key_id`, the full canonical action **and its digest**, `policy_version`, issue + expiry timestamps. One-time consumption is tracked at the chain level.
- **action receipt** — agent identity, `run_id`, the same canonical action **digest**, execution outcome + timestamp, and `parent_approval_ref` (content-derived id of the approval).

The shared `action_digest` field is the join the binding depends on.

In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band; the verifier NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"ha-key-1":    b64u(approver_sk.verify_key.encode())}
AGENT_KEYS    = {"agent-key-1": b64u(agent_sk.verify_key.encode())}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_envelope(payload, sk):
    # One envelope for BOTH receipt kinds: {payload, sig}. key_id lives INSIDE the
    # signed payload, so an attacker cannot re-point a receipt at another key.
    return {"payload": payload, "sig": b64u(sk.sign(canon(payload)).signature)}

def verify_envelope(receipt, expected_typ, trusted_keys):
    """The SHARED verifier contract. Structural + signature checks for any receipt
    kind; the caller picks which pinned registry (authority) resolves key_id.
    Fails closed on ANY attacker-shaped input: malformed is a refusal, not a crash."""
    try:
        payload, sig = receipt["payload"], receipt["sig"]
        signed_bytes, sig_bytes = canon(payload), b64u_dec(sig)
    except (KeyError, TypeError, ValueError, base64.binascii.Error):
        return (False, "receipt malformed (missing or unreadable fields)")
    if not isinstance(payload, dict):
        return (False, "receipt malformed (payload is not an object)")
    if payload.get("typ") != expected_typ:
        return (False, f"wrong receipt type (expected {expected_typ})")
    # Key freshness is part of authority: a key_id that has been rotated out of the
    # pinned registry no longer confers anything, even with a valid signature.
    pub = trusted_keys.get(payload.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {payload.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # CryptoError is the base of BadSignatureError AND the wrong-length ValueError,
    # so forged and malformed signatures both refuse cleanly.
    try:
        VerifyKey(b64u_dec(pub)).verify(signed_bytes, sig_bytes)
    except CryptoError:
        return (False, "signature invalid (forged or tampered)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at_ms, sk=approver_sk,
                   key_id="ha-key-1", policy_version=None, ttl_ms=15 * 60 * 1000):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "typ": "human-approval",
        "approver_id": approver_id,
        "key_id": key_id,
        "action": approved_action,                          # the FULL canonical action
        "action_digest": sha256_hex(canon(approved_action)),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at_ms": approved_at_ms,
        "expires_at_ms": approved_at_ms + ttl_ms,
    }
    return sign_envelope(payload, sk)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", 1_750_000_000_000)
print(verify_envelope(approval, "human-approval", APPROVER_KEYS))
print("binds digest:", approval["payload"]["action_digest"][:16], "…  under", approval["payload"]["policy_version"])

(True, 'envelope ok')
binds digest: e68137d4f7788a7d …  under refunds-v3


## `verify_chain`: where the binding is actually decided

`verify_chain` is **not** a convenience wrapper over two signature checks. It is the one place where the shared canonical `action_digest`, the policy/key/expiry **freshness** of the approval, and the **one-time consumption** of the approval are checked together, against the action being executed *right now*.

Each failure mode refuses with a **distinct reason**, so a future reader of the refusal can tell whether authority went stale (policy moved, key rotated, approval expired, approval already consumed) or the executed action changed out from under a still-valid approval (digest substitution).

In [5]:
def approval_ref(receipt):                     # content-derived id of the approval
    return "ha_" + sha256_hex(canon(receipt["payload"]))   # full SHA-256 (no truncation)

def agent_receipt(action, approval, executed_at_ms, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "typ": "agent-action",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "key_id": key_id,
        "action": executed_action,
        "action_digest": sha256_hex(canon(executed_action)),  # same join field
        "parent_approval_ref": approval_ref(approval),
        "outcome": "performed",
        "executed_at_ms": executed_at_ms,
    }
    return sign_envelope(payload, sk)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now_ms):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human-approval", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent-action", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")
    a, g = approval["payload"], agent_rcpt["payload"]

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_hex(canon(action_being_executed))
    if a.get("action_digest") != executing_digest or a.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if g.get("action_digest") != executing_digest or g.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if g.get("parent_approval_ref") != approval_ref(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if a.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {a.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    if not isinstance(a.get("expires_at_ms"), int) or now_ms >= a["expires_at_ms"]:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = approval_ref(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {a['approver_id']}, executed by {g['agent_id']}")

def execute(action, approval, agent_rcpt, now_ms):
    ok, why = verify_chain(action, approval, agent_rcpt, now_ms)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, 1_750_000_000_500)
print(execute(action, approval, receipt, now_ms=1_750_000_000_600))

(True, 'executed')


## What the binding catches

Every case below fails **closed** with a **distinct reason**. The first block is the classic set (tamper, confused deputy, replay, forgery on either authority, malformed input). The second block is the pair that makes the property real rather than asserted:

- **stale authority** — the signature is still valid, but the policy version moved, the approver key was rotated out of the pinned registry, or the approval expired before execution;
- **digest substitution** — a validly signed action receipt whose `parent_approval_ref` points at a *real* approval, but that approval's canonical action digest does not match the action actually being executed.

In [6]:
NOW = 1_750_000_001_000

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW)

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"payload": {}}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
print("wrong-len-sig-appr  ->", verify_chain(action, {**fresh, "sig": "AAAA"}, agent_receipt(action, fresh, NOW), NOW))
print("wrong-len-sig-agent ->", verify_chain(action, fresh, {**agent_receipt(action, fresh, NOW), "sig": "AAAA"}, NOW))

# 9. non-dict payload: a list payload refuses cleanly instead of raising AttributeError.
print("nondict-payload     ->", verify_chain(action, {"payload": [1, 2], "sig": "AAAA"}, agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("ha-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["ha-key-1"] = rotated_out           # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", NOW - 3_600_000, ttl_ms=60_000)
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW)
substituted = agent_receipt(action, approval_b, NOW)     # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged or tampered)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged or tampered)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (missing or unreadable fields)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (missing or unreadable fields)')
wrong-len-sig-appr  -> (False, 'approval: signature invalid (forged or tampered)')
wrong-len-sig-agent -> (False, 'agent receipt: signature invalid (forged or tamp

## What this proves — and what it doesn't

**Proves:** a named human approved *this exact canonical action* (full action + digest, signed with a key pinned out of band), and the agent executed *exactly that approved action* (same digest, receipt bound to the approval by content-derived reference) — while the approval's policy version, key, and expiry were still current, exactly once. If either side changes, the chain fails closed, and the refusal reason tells you **which** property broke: stale authority vs. a changed action.

**Doesn't prove:** that the approval UI showed the human what they thought they were signing (WYSIWYS is its own problem), that the key wasn't coerced or stolen before rotation, or that downstream effects matched the action. Signed ≠ authorized: a valid signature over a stale policy, a rotated key, an expired window, or a different digest confers nothing here.

The two receipt kinds share one envelope and one `verify_chain` code path on purpose — the same shape the lesson's action receipts use — so the binding a learner builds in the main notebook is the same code that checks the human's approval. One verifier contract, separate authorities, joined by the canonical action digest and nothing else.